In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import  load_dotenv

In [ ]:
load_dotenv()

In [ ]:
#Defining the ParentState
class ParentState(TypedDict):
    question:str
    answer_english : str
    answer_hindi:str

In [ ]:
parent_llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash")
subgraph_llm = ChatGoogleGenerativeAI(model = "gemini-3.5-flash")

In [ ]:
def translate_text(state: ParentState):

    prompt = f"""
    Translate the following text to Hindi.
    Keep it natural and clear. Do not add extra content.
    Text:
    {state["answer_eng"]}
    """.strip()
    translated_text = subgraph_llm.invoke(prompt).content
    return {'answer_hin': translated_text}

In [ ]:
#defining subgraph with the same parent state
subgraph_builder = StateGraph(ParentState)

subgraph_builder.add_node('translate_text',translate_text)

subgraph_builder.add_edge(START,'translate_text')
subgraph_builder.add_edge('translate_text',END)

subgraph = subgraph_builder.compile()

In [ ]:
subgraph

In [ ]:
def generate_answer(state: ParentState):
    answer = parent_llm.invoke(f"You are a helpful assistant. Answer clearly.\n\nQuestion: {state['question']}").content
    return {'answer_eng': answer}

In [ ]:
parent_builder = StateGraph(ParentState)

parent_builder.add_node("answer",generate_answer)
#adding subgraph inside the same node with the share parent state
parent_builder.add_node('translate',subgraph)

parent_builder.add_edge(START,'answer')
parent_builder.add_edge('answer','translate')
parent_builder.add_edge('translate',END)

graph = parent_builder.compile()

In [ ]:
graph

In [ ]:
graph.invoke({'question': 'What is quantum physics'})